# Phase 5 — GRPO tracer (seed 3407): the main event begins

**Runtime: A100 recommended** (dev-slice work; ~1.5–2 h, ~20–25 units. L4 works
but ~2× slower). This is the single most expensive notebook so far — the tracer
exists so we find the failure modes on ONE seed before paying for three.

**What GRPO does, in one line:** the model writes 8 candidate fixes per bug, we
EXECUTE the tests, and the fixes that pass get pushed up relative to their
groupmates that fail — trial and error with an honest judge, on-policy.

**Design (all pre-committed here):**
- Init: merged SFT seed 3407 + fresh LoRA r32/α64 (Amendment A3 paired lineage);
  KL anchor = the merged SFT model (adapter-disabled reference)
- Data: the RL pile only (376 mixed-outcome bugs — where the learning signal is)
- Reward: the repo's `src/reward.py` — execution-verified, CoRPO invariant
  (no failing patch can ever out-reward a passing one), unit-tested
- **Hardened runner:** tests print a RANDOM run-time sentinel after they all
  execute. A policy that learns `sys.exit(0)`-style early exits gets exit code 0
  but no sentinel → scored as failing. (Meta's agentic-repair study watched
  models delete validation code; we assume ours will try.)
- Pre-flight: variance gate ON THE INIT (the base-model gate passed at 69%,
  but SFT solves more — if the pile went all-easy, GRPO has no signal)
- 250 steps, group size 8, temp 1.0, lr 5e-6, KL β 0.04, no vLLM (robustness)
- Hacking watch: rewards + sample completions stream to Drive during training

**Verdict after:** dev pass@1 vs SFT (58.7) and DPO (61.3) on this lineage —
GRPO must convert headroom (pass@16 91.8) into pass@1.

In [ ]:
# Setup: Drive, repo, canonical modules (prompt + reward + variance gate)
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib, gc, uuid
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
PHASE5 = '/content/drive/MyDrive/rl-post-training/phase5'
os.makedirs(PHASE5, exist_ok=True)
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts', 'reward', 'variance_gate'):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
import reward as rw
import variance_gate as vg
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
restraint = json.load(open('/content/ptl/data/data_v0_restraint.json'))
routing = {r['id']: r['pile'] for r in json.load(open(f'{PHASE2}/routing_v0.json'))}
train_bugs = [b for b in bugs if b['split'] == 'train']
rl_bugs = [b for b in train_bugs if routing.get(b['id']) == 'rl']
dev_bugs = [b for b in bugs if b['split'] == 'dev']
dev_clean = [r for r in restraint if r['split'] == 'dev']
SEED = 3407
print(len(rl_bugs), 'RL-pile bugs |', len(dev_bugs), 'dev |', len(dev_clean), 'dev clean')

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y -q torchao torchaudio torchvision timm

In [ ]:
# HARDENED sandboxed runner + the canonical reward
import subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor
SENTINEL = uuid.uuid4().hex   # fresh every run; never appears in any file or prompt

def run_tests(code, rec, timeout=10):
    """Execute candidate against the FULL suite. Returns (compiles, n_passed, n_tests).
    Early-exit hacks (sys.exit(0) etc.) yield exit code 0 but NO sentinel -> 0 passed."""
    tests = list(rec['test_list'])
    try:
        compile(code, '<cand>', 'exec'); compiles = True
    except Exception:
        return (False, 0, len(tests))
    harness = (
        '\n'.join(list(rec['test_imports'])) + '\n' + code + '\n'
        + '__zz_pass = 0\n'
        + f'for __zz_t in {tests!r}:\n'
        + '    try:\n        exec(__zz_t)\n        __zz_pass += 1\n'
        + '    except BaseException:\n        pass\n'
        + f'print("{SENTINEL}", __zz_pass)\n'
    )
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(harness); path = f.name
    try:
        r = subprocess.run([sys.executable, path], capture_output=True, timeout=timeout)
        n_passed = 0
        for line in r.stdout.decode(errors='ignore').splitlines():
            parts = line.split()
            if len(parts) == 2 and parts[0] == SENTINEL and parts[1].isdigit():
                n_passed = int(parts[1])   # no sentinel line = hack or crash = 0
    except subprocess.TimeoutExpired:
        n_passed = 0
    finally:
        os.unlink(path)
    return (compiles, n_passed, len(tests))

HACK_LOG = f'{PHASE5}/hacking_watch_s{SEED}.jsonl'
_call_n = [0]

def grpo_reward(prompts, completions, test_imports, test_list, **kw):
    recs = [{'test_imports': ti, 'test_list': tl} for ti, tl in zip(test_imports, test_list)]
    codes = [extract_code(c) for c in completions]
    with ThreadPoolExecutor(max_workers=8) as pool:
        outs = list(pool.map(lambda a: run_tests(*a), zip(codes, recs)))
    rewards = [rw.reward(rw.ExecResult(compiles=c, num_passed=p, num_tests=t,
                                       output_tokens=len(comp) // 4))
               for (c, p, t), comp in zip(outs, completions)]
    _call_n[0] += 1
    best = max(range(len(rewards)), key=lambda i: rewards[i])
    worst = min(range(len(rewards)), key=lambda i: rewards[i])
    with open(HACK_LOG, 'a') as f:
        f.write(json.dumps({'call': _call_n[0], 'mean_r': sum(rewards)/len(rewards),
                            'n_pass': sum(1 for r in rewards if r >= 1.25),
                            'best_snippet': codes[best][:200],
                            'worst_snippet': codes[worst][:200]}) + '\n')
    return rewards

# self-test: a correct patch, a broken patch, and an early-exit HACK must score sanely
_demo = {'test_imports': [], 'test_list': ['assert add2(2) == 4', 'assert add2(0) == 2']}
_good = 'def add2(x):\n    return x + 2'
_bad = 'def add2(x):\n    return x'
_hack = 'import sys\nsys.exit(0)\ndef add2(x):\n    return x'
for name, cand in (('good', _good), ('bad', _bad), ('HACK sys.exit(0)', _hack)):
    c, p, t = run_tests(cand, _demo)
    print(f"{name:<18} compiles={c} passed={p}/{t} reward={rw.reward(rw.ExecResult(c, p, t)):.2f}")
print('\nHACK must show passed=0. CoRPO bounds:', rw.max_failing_reward(), '<', rw.min_passing_reward())

In [ ]:
# Load the init: merged SFT seed 3407 + fresh LoRA (KL ref = merged SFT)
import torch
from unsloth import FastLanguageModel
merged = f'/content/merged_s{SEED}'
if not os.path.exists(merged):
    m, t = FastLanguageModel.from_pretrained(
        f'{PHASE3}/sft_notrace_s{SEED}_ep1', max_seq_length=1536,
        load_in_4bit=False, dtype=torch.bfloat16)
    m.save_pretrained_merged(merged, t, save_method='merged_16bit')
    del m, t; gc.collect(); torch.cuda.empty_cache()
model, tok = FastLanguageModel.from_pretrained(
    merged, max_seq_length=1536, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=64, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED)

def chat(p):
    return tok.apply_chat_template([{'role': 'user', 'content': p}],
                                   tokenize=False, add_generation_prompt=True)
print('init loaded')

In [ ]:
# PRE-FLIGHT: variance gate ON THE INIT (40 random RL-pile bugs x G=8, temp 1.0)
@torch.no_grad()
def sample_k(source_code, test_list, k=8):
    enc = tok([chat(build_repair_prompt(source_code, test_list))],
              return_tensors='pt', padding=True, padding_side='left').to('cuda')
    out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                         num_return_sequences=k, max_new_tokens=512,
                         pad_token_id=tok.eos_token_id)
    return [extract_code(t) for t in tok.batch_decode(
        out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]

FastLanguageModel.for_inference(model)
probe_bugs = random.Random(SEED).sample(rl_bugs, 40)
pass_counts = []
for i, b in enumerate(probe_bugs):
    codes = sample_k(b['buggy'], b['test_list'])
    with ThreadPoolExecutor(max_workers=8) as pool:
        outs = list(pool.map(lambda c: run_tests(c, b), codes))
    pass_counts.append((b['id'], sum(1 for c_, p, t in outs if p == t)))
    if (i + 1) % 10 == 0:
        print(f'{i+1}/40 probed')
g = vg.gate(pass_counts, group_size=8)
print(f"\nGATE (on the SFT init): learnable {g['learnable_fraction']*100:.0f}%  (needs >=30%)")
print('PASS — GRPO has fuel. Proceed.' if g['pass_gate'] else
      'FAIL — STOP. The pile is degenerate for this init. Report before training.')

In [ ]:
# TRAIN: GRPO, 250 steps (~2.7 passes over the RL pile), rewards stream to Drive
try:
    from unsloth import PatchFastRL
    PatchFastRL('GRPO', FastLanguageModel)
except ImportError:
    pass
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

rows = [{'prompt': chat(build_repair_prompt(b['buggy'], b['test_list'])),
         'test_imports': list(b['test_imports']), 'test_list': list(b['test_list'])}
        for b in rl_bugs]
ds = Dataset.from_list(rows).shuffle(seed=SEED)
FastLanguageModel.for_training(model)
cfg = GRPOConfig(
    per_device_train_batch_size=8, gradient_accumulation_steps=4,
    num_generations=8, max_prompt_length=768, max_completion_length=512,
    temperature=1.0, top_p=0.95, beta=0.04,
    learning_rate=5e-6, lr_scheduler_type='cosine', warmup_ratio=0.1,
    max_steps=250, logging_steps=5, seed=SEED, output_dir='/content/out',
    report_to='none', save_strategy='no')
trainer = GRPOTrainer(model=model, processing_class=tok, reward_funcs=grpo_reward,
                      args=cfg, train_dataset=ds)
trainer.train()
model.save_pretrained(f'{PHASE5}/grpo_s{SEED}')
print('adapter saved to Drive phase5/. WATCH the reward column above:')
print('healthy = mean reward climbing, KL bounded. Suspicious = sudden reward jump')
print('with weird completions in', HACK_LOG)

In [ ]:
# VERDICT: dev eval + restraint probe + three-arm comparison on this lineage
def passes_all(code, rec):
    c, p, t = run_tests(code, rec)
    return p == t

def dev_eval(tag, k=16):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for b in dev_bugs:
        codes = sample_k(b['buggy'], b['test_list'], k=k)
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes_all(c, b), codes))
        per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    p16 = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    res = {'tag': tag, 'pass@1': p1, 'pass@16': p16, 'gap': p16 - p1, 'per_bug': per_bug}
    json.dump(res, open(f'{PHASE5}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@16={p16*100:.1f}%  gap={(p16-p1)*100:.1f}")
    return res

def restraint_probe(tag, k=4):
    FastLanguageModel.for_inference(model)
    norm = lambda s: ' '.join(s.split())
    sp = un = tot = 0
    for r in dev_clean:
        codes = sample_k(r['code'], r['test_list'], k=k)
        with ThreadPoolExecutor(max_workers=8) as pool:
            flags = list(pool.map(lambda c: passes_all(c, r), codes))
        sp += sum(flags); un += sum(norm(c) == norm(r['code']) for c in codes); tot += k
    json.dump({'tag': tag, 'still_pass': sp/tot, 'unchanged': un/tot},
              open(f'{PHASE5}/restraint_probe_{tag}.json', 'w'))
    print(f"[probe {tag}]  still passes: {sp/tot*100:.1f}%  unchanged: {un/tot*100:.1f}%")

res = dev_eval(f'grpo_seed{SEED}')
restraint_probe(f'grpo_s{SEED}')
print()
print(f"{'arm (seed 3407 lineage)':<26} pass@1   pass@16   gap")
sft = json.load(open(f'{PHASE3}/dev_eval_ep1_seed{SEED}.json'))
dpo = json.load(open(f'/content/drive/MyDrive/rl-post-training/phase4/dev_eval_dpo_ep1_seed{SEED}.json'))
for name, r in (('SFT', sft), ('SFT+DPO', dpo), ('SFT+GRPO', res)):
    print(f"{name:<26} {r['pass@1']*100:6.1f}%  {r['pass@16']*100:6.1f}%  {r['gap']*100:5.1f}")
print('\nGRPO thesis: convert headroom into pass@1 WITHOUT collapsing pass@16.')
print('Ruler: < ~1.5 pts = tie. Bring the whole printout + the reward curve back.')

## Bring back to the session
1. The **gate line** (learnable % on the SFT init)
2. The **reward trajectory** from the training table (first/middle/last few rows
   — reward mean, KL, completion length)
3. The **verdict table** (SFT vs SFT+DPO vs SFT+GRPO on this lineage)
4. The **restraint probe** line
5. Anything weird in the hacking watch (sudden reward jumps, degenerate snippets)

If the tracer is healthy: seeds 42 + 1234 (same recipe), then the random-reward
control, then the Phase-7 final exam for all arms.